## Neccessary Libraries

In [1]:
import numpy as np

## Trapezoidal Rule

In [2]:
def Trapezoid_Rule(n,f,a,b): 
    h = (b-a) / n
    summation = 0
    for i in range(1,n): 
        summation += f(a + i*h)
    integral = h / 2 * (f(a) + 2*summation + f(b))
    n_evals = n + 1
    return integral, n_evals

## Adaptive Trapezoidal Rule

In [3]:
def Adaptive_Trapezoid(m,f,a,b,tol): 
    I_coarse, _ = Trapezoid_Rule(m, f, a, b)
    total_eval = m + 1

    while True: 
        m = 2 * m
        I_fine, evals = Trapezoid_Rule(m, f, a, b)
        total_eval += evals

        if abs(I_fine - I_coarse) < tol: 
            return I_fine, m, total_eval
        I_coarse = I_fine
    return I_coarse, total_eval

## Coefficients of Legendre polynomials

In [4]:
def Legendre_Coeff(n): 
    # Coefficients of Legendre polynomials
    # [A,B] are the alpha(k) and beta(k) coefficients 
    # for the Legendre Polynomial of degree N
    if n < 1:
        raise ValueError('n must be >= 1') 

    a = np.zeros(n)
    b = np.zeros(n)
    b[0] = 2

    ## Double check this 
    for k in range(1,n):
        b[k] = k**2 / (4*k**2 - 1)

    return a, b

## Coefficients of Gauss-Legendre Form

In [5]:
def GL_Coeff(n): 
    # Gauss-Legendre formula
    # [X,W] are the nodes and the weights of the
    # Gauss-Legendre formula with N nodes

    if n < 1:
        raise ValueError('n must be >= 1') 

    a, b = Legendre_Coeff(n)
    JacM = np.diag(a) + np.diag(np.sqrt(b[1:n]),1) + np.diag(np.sqrt(b[1:n]),-1)
    eigenvalues, eigenvectors = np.linalg.eigh(JacM)
    x = eigenvalues
    w = b[0] * eigenvectors[0, :]**2
    ind = np.argsort(x)
    x = x[ind]
    w = w[ind]
    
    return x, w

## Guass-Legandre Quadrature 

In [6]:
def GL_Quadrature(n, f, a, b, m): 
    t, w = GL_Coeff(n)
    if m == 0: 
        mid = (a+b) / 2
        half = (b-a) / 2
        x_nodes = half * t + mid
        integral = half * np.dot(w, f(x_nodes))
        n_evals = n

    else:
        h = (b-a) / m
        half_h = h / 2
        total = 0

        for k in range(m):
            a_k = a + k * h
            mid_k = a_k + half_h
            x_nodes = half_h * t + mid_k
            total += np.dot(w, f(x_nodes))

        integral = half_h * total
        n_evals = n * m

    return integral, n_evals

## Test Functions & Exact Values

In [7]:
f1 = lambda x: np.exp(x) 
f2 = lambda x: np.exp(np.sin(2*x)) * np.cos(2*x) 
f3 = lambda x: np.tanh(x) 
f4 = lambda x: x * np.cos(2*np.pi*x) 
f5 = lambda x: x + (1 / x) 
functions = [f1, f2, f3, f4, f5]

f1_exact = np.exp(3) - 1 
f2_exact = 0.5 * (-1 + np.exp(np.sqrt(3) / 2))
f3_exact = np.log(np.cosh(1) / np.cosh(2))
f4_exact = -1 / (2 * np.pi**2)
f5_exact = (2.5**2 - 0.1**2) / 2 + np.log(2.5 / 0.1)
exact = [f1_exact, f2_exact, f3_exact, f4_exact, f5_exact]

## Task 1 - Estimate $H_m$

In [30]:
def second_derivative_max(f, a, b, N = 1000): 
    x = np.linspace(a,b,N)
    h = x[1] - x[0]
    f2 = np.abs(np.diff(f(x),2)) / h**2 
    return np.max(f2)

def subinterval_bound(f, a, b, tol): 
    second_derivative = second_derivative_max(f, a, b,)
    H_m = np.sqrt(12 * tol / ((b - a) * second_derivative))
    n = int(np.ceil((b-a)/ H_m))
    return H_m, n, second_derivative

tols = [1e-2, 1e-4, 1e-6, 1e-8]
names = ['f1', 'f2', 'f3', 'f4', 'f5']
intervals = [(0,3), (0,np.pi/3), (-2,1), (0,3.5), (0.1, 2.5)]

for fname, f, (a,b) in zip(names, functions, intervals):
    print(f"\n{fname}  [{a}, {b:.4f}]")
    print(f"  {'tol':>8}  {'max|f´´|':>12}  {'Hm':>12}  {'n':>6}")
    print(f"  {'-'*8}  {'-'*12}  {'-'*12}  {'-'*6}")
    for tol in tols:
        Hm, n, M2 = subinterval_bound(f, a, b, tol)
        print(f"  {tol:>8.0e}  {M2:>12.4f}  {Hm:>12.6f}  {n:>6}")


f1  [0, 3.0000]
       tol      max|f´´|            Hm       n
  --------  ------------  ------------  ------
     1e-02       20.0253      0.044693      68
     1e-04       20.0253      0.004469     672
     1e-06       20.0253      0.000447    6713
     1e-08       20.0253      0.000045   67125

f2  [0, 1.0472]
       tol      max|f´´|            Hm       n
  --------  ------------  ------------  ------
     1e-02       16.2830      0.083890      13
     1e-04       16.2830      0.008389     125
     1e-06       16.2830      0.000839    1249
     1e-08       16.2830      0.000084   12484

f3  [-2, 1.0000]
       tol      max|f´´|            Hm       n
  --------  ------------  ------------  ------
     1e-02        0.7698      0.227951      14
     1e-04        0.7698      0.022795     132
     1e-06        0.7698      0.002280    1317
     1e-08        0.7698      0.000228   13161

f4  [0, 3.5000]
       tol      max|f´´|            Hm       n
  --------  ------------  ------------

## Task 2 - Obeserved & True Error

## Individual Testing (Later)

In [24]:
# Pick a function and its interval
f = lambda x: np.exp(x)
a, b = 0, 3
exact = np.exp(3) - 1          

# n = 2 always 
approx, n_evals = GL_Quadrature(2, f, a, b, m=10)

print(f"Approx: {approx}")
print(f"Exact:  {exact}")
print(f"Error:  {abs(approx - exact)}")
print(f"Evals:  {n_evals}\n")

# Trapezoidal Rule
approx, n_evals = Trapezoid_Rule(2, f, a, b)

print(f"Approx: {approx}")
print(f"Exact:  {exact}")
print(f"Error:  {abs(approx - exact)}")
print(f"Evals:  {n_evals}\n")

# adaptive Trap Rule
tol = 1e-6  
approx, m_final, n_evals = Adaptive_Trapezoid(2, f, a, b, tol)

print(f"Approx: {approx}")
print(f"Exact:  {exact}")
print(f"Error:  {abs(approx - exact)}")
print(f"Subint: {m_final}")
print(f"Evals:  {n_evals}\n")


Approx: 19.08550123980601
Exact:  19.085536923187668
Error:  3.5683381657491964e-05
Evals:  20

Approx: 22.536686297897845
Exact:  19.085536923187668
Error:  3.4511493747101767
Evals:  3

Approx: 19.085537136485172
Exact:  19.085536923187668
Error:  2.1329750410359338e-07
Subint: 8192
Evals:  16395

